# Corpus Builder — Morocco-focused Arabic Wikipedia corpus

Replaces the synthetic 80-passage pilot corpus with ~3,000 real Arabic passages, so retrieval is a realistic task rather than an 80-way choice.

**Run this in Colab** — it needs open internet access to reach the Wikipedia API.

Output: `corpus_v2.json`, containing the original 80 pilot passages (tagged `pilot_synthetic`) plus the new Wikipedia passages (tagged `wikipedia_ar`), kept separable for honest reporting in the paper.

### Install dependencies

In [ ]:
# !pip install -q requests tqdm

### Config

In [ ]:
CONFIG = {
    "lang": "ar",
    "target_passages": 3000,
    "min_passage_chars": 300,      # too-short passages make poor retrieval targets
    "max_passage_chars": 900,      # keeps passages focused enough to be a single answer source
    "max_articles": 600,           # cap on articles fetched; each yields several passages
    "seed_categories": [
        "تصنيف:المغرب",
        "تصنيف:مدن_المغرب",
        "تصنيف:اقتصاد_المغرب",
        "تصنيف:تاريخ_المغرب",
        "تصنيف:ثقافة_المغرب",
        "تصنيف:جغرافيا_المغرب",
        "تصنيف:سياسة_المغرب",
        "تصنيف:التعليم_في_المغرب",
        "تصنيف:مجتمع_مغربي",
        "تصنيف:رياضة_في_المغرب",
        "تصنيف:مطبخ_مغربي",
        "تصنيف:سياحة_في_المغرب",
    ],
    "category_depth": 2,           # how many levels of subcategories to traverse
    "user_agent": "DialectAwareArabicRAG/1.0 (research project; contact: your-email@example.com)",
}

# NOTE: set a real contact email in user_agent above — Wikimedia's API policy
# asks for an identifiable user agent, and requests may be throttled without one.

### Collect article titles by traversing Morocco-related categories

In [ ]:
import requests, time
from tqdm.auto import tqdm

API = f"https://{CONFIG['lang']}.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": CONFIG["user_agent"]}

def api_get(params, retries=3):
    for attempt in range(retries):
        try:
            r = requests.get(API, params={**params, "format": "json"}, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == retries - 1:
                print(f"  API call failed after {retries} tries: {e}")
                return {}
            time.sleep(2 * (attempt + 1))
    return {}

def category_members(category, cmtype="page", limit=500):
    """Yield member titles of a category ('page' for articles, 'subcat' for subcategories)."""
    members, cont = [], None
    while True:
        params = {
            "action": "query", "list": "categorymembers", "cmtitle": category,
            "cmtype": cmtype, "cmlimit": limit,
        }
        if cont:
            params["cmcontinue"] = cont
        data = api_get(params)
        batch = data.get("query", {}).get("categorymembers", [])
        members.extend(m["title"] for m in batch)
        cont = data.get("continue", {}).get("cmcontinue")
        if not cont:
            break
        time.sleep(0.2)
    return members

article_titles, seen_cats = set(), set()
frontier = [(c, 0) for c in CONFIG["seed_categories"]]

print("Traversing Morocco-related categories...")
with tqdm(total=CONFIG["max_articles"]) as pbar:
    while frontier and len(article_titles) < CONFIG["max_articles"]:
        cat, depth = frontier.pop(0)
        if cat in seen_cats:
            continue
        seen_cats.add(cat)

        before = len(article_titles)
        for title in category_members(cat, cmtype="page"):
            if len(article_titles) >= CONFIG["max_articles"]:
                break
            article_titles.add(title)
        pbar.update(len(article_titles) - before)

        if depth < CONFIG["category_depth"]:
            for sub in category_members(cat, cmtype="subcat"):
                if sub not in seen_cats:
                    frontier.append((sub, depth + 1))
        time.sleep(0.2)

article_titles = sorted(article_titles)
print(f"\nCollected {len(article_titles)} article titles from {len(seen_cats)} categories.")

### Fetch article plain text

In [ ]:
def fetch_extract(title):
    """Fetch the plain-text extract of one article."""
    data = api_get({
        "action": "query", "prop": "extracts", "explaintext": 1,
        "exsectionformat": "plain", "titles": title, "redirects": 1,
    })
    pages = data.get("query", {}).get("pages", {})
    for _, page in pages.items():
        return page.get("extract", "") or ""
    return ""

articles = {}
print("Fetching article text...")
for title in tqdm(article_titles):
    text = fetch_extract(title)
    if text and len(text) >= CONFIG["min_passage_chars"]:
        articles[title] = text
    time.sleep(0.15)  # be polite to the API

print(f"\nFetched usable text for {len(articles)} articles.")

### Clean and chunk into retrieval passages

In [ ]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r"=+\s*[^=]+\s*=+", " ", text)          # strip section headings
    text = re.sub(r"\[\d+\]", " ", text)                   # strip footnote markers
    text = re.sub(r"\{\{.*?\}\}", " ", text, flags=re.S)   # strip leftover templates
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_sentences(text: str):
    # Arabic full stop, question mark, exclamation; keep the delimiter
    parts = re.split(r"(?<=[.!؟?])\s+", text)
    return [p.strip() for p in parts if p.strip()]

def chunk_article(text: str, min_chars: int, max_chars: int):
    """Greedily pack sentences into passages within the size window."""
    passages, buf = [], ""
    for sent in split_sentences(clean_text(text)):
        if len(buf) + len(sent) + 1 <= max_chars:
            buf = f"{buf} {sent}".strip()
        else:
            if len(buf) >= min_chars:
                passages.append(buf)
            buf = sent if len(sent) <= max_chars else sent[:max_chars]
    if len(buf) >= min_chars:
        passages.append(buf)
    return passages

wiki_chunks = []
for title, text in articles.items():
    for j, passage in enumerate(chunk_article(text, CONFIG["min_passage_chars"], CONFIG["max_passage_chars"])):
        wiki_chunks.append({
            "chunk_id": f"wiki_{len(wiki_chunks)+1:05d}",
            "text": passage,
            "source": "wikipedia_ar",
            "article_title": title,
            "passage_index": j,
        })
    if len(wiki_chunks) >= CONFIG["target_passages"]:
        break

wiki_chunks = wiki_chunks[: CONFIG["target_passages"]]
print(f"Built {len(wiki_chunks)} Wikipedia passages from {len(set(c['article_title'] for c in wiki_chunks))} articles.")
print("\nExample passage:")
print(wiki_chunks[0]["article_title"], "->", wiki_chunks[0]["text"][:200], "...")

### Deduplicate near-identical passages

In [ ]:
from collections import OrderedDict

def dedup_key(text: str) -> str:
    return re.sub(r"\s+", "", text)[:200]  # crude but effective for exact/near repeats

deduped = OrderedDict()
for c in wiki_chunks:
    k = dedup_key(c["text"])
    if k not in deduped:
        deduped[k] = c

wiki_chunks = list(deduped.values())
# Reassign stable sequential ids after dedup
for i, c in enumerate(wiki_chunks, start=1):
    c["chunk_id"] = f"wiki_{i:05d}"

print(f"After deduplication: {len(wiki_chunks)} passages.")

### Merge with the existing pilot corpus (keeps your 300 QA items valid)

In [ ]:
import json, requests as rq

REPO_RAW = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main"
pilot_corpus = json.loads(rq.get(f"{REPO_RAW}/data/corpus.json").text)

# Tag the pilot passages so the two provenances stay clearly separable.
# This matters for the paper: the 80 pilot passages are author-written, not
# real-world text, and that must be disclosed rather than blended in silently.
for c in pilot_corpus:
    c["source"] = "pilot_synthetic"

merged_corpus = pilot_corpus + wiki_chunks

with open("corpus_v2.json", "w", encoding="utf-8") as f:
    json.dump(merged_corpus, f, ensure_ascii=False, indent=2)

print(f"Merged corpus: {len(merged_corpus)} passages "
      f"({len(pilot_corpus)} pilot_synthetic + {len(wiki_chunks)} wikipedia_ar)")

### Summary stats

In [ ]:
from collections import Counter

lengths = [len(c["text"]) for c in merged_corpus]
sources = Counter(c["source"] for c in merged_corpus)

print("Corpus summary")
print("-" * 40)
print(f"Total passages:      {len(merged_corpus)}")
for src, n in sources.items():
    print(f"  {src:<20} {n}")
print(f"Mean passage length: {sum(lengths)/len(lengths):.0f} chars")
print(f"Min / Max length:    {min(lengths)} / {max(lengths)} chars")
print(f"Unique articles:     {len(set(c.get('article_title','') for c in wiki_chunks))}")

### Download the new corpus

In [ ]:
from google.colab import files
files.download("corpus_v2.json")

# Next steps after this notebook:
#   1. Commit corpus_v2.json to the GitHub repo (data/corpus_v2.json).
#   2. Re-run the retrieval pipeline against it — retrieval is now a ~3,000-way
#      task instead of 80-way, so expect Recall@k to drop across ALL conditions.
#      The number that matters is the GAP between conditions, not absolute values.
#   3. Generate new QA items grounded in the Wikipedia passages to grow the
#      benchmark beyond the existing 300 pilot items.